# Engenharia Econômica — Aula 6: Análise de Investimentos

**Base:** *Engenharia Econômica - Aula 6 - Análise de Investimentos.pdf* — Prof. Dr. Octaviano Rojas Luiz.

Este notebook reproduz os conceitos centrais e resolve os **Exercícios 1 a 9** da aula.

## Conteúdo
- Taxa Mínima de Atratividade (TMA)
- Valor Presente Líquido (VPL)
- Projetos mutuamente excludentes
- VAE / CAUE
- Taxa Interna de Retorno (TIR)
- TIR incremental
- Payback simples e descontado
- Valor residual
- Sensibilidade à TMA
- Substituição de ativos

## 1. Fórmulas principais

### Valor Presente Líquido
\[
VPL=-I_0+\sum_{t=1}^{n}\frac{FC_t}{(1+TMA)^t}
\]

Para fluxos uniformes:
\[
VPL=-I_0+FC\cdot\frac{1-(1+TMA)^{-n}}{TMA}
\]

### Valor Anual Equivalente / CAUE
\[
VAE=VPL\cdot\frac{i(1+i)^n}{(1+i)^n-1}
\]

### TIR
É a taxa que satisfaz:
\[
VPL(TIR)=0
\]

### Payback descontado
Primeiro descontamos os fluxos pela TMA e depois acumulamos até recuperar o investimento.

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt

def vpl(taxa, fluxos):
    """fluxos[0] é o fluxo do momento zero."""
    return sum(fc / (1 + taxa)**t for t, fc in enumerate(fluxos))

def fator_vp_serie(taxa, n):
    return (1 - (1 + taxa)**(-n)) / taxa

def fator_recuperacao_capital(taxa, n):
    return taxa * (1 + taxa)**n / ((1 + taxa)**n - 1)

def vae(valor_presente, taxa, n):
    return valor_presente * fator_recuperacao_capital(taxa, n)

def tir_bissecao(fluxos, minimo=-0.9999, maximo=10.0, tol=1e-12, max_iter=1000):
    """TIR por busca de intervalo + bisseção; adequada aos fluxos convencionais desta aula."""
    def f(r):
        return vpl(r, fluxos)

    # procurar um intervalo com troca de sinal
    passos = 20000
    anterior_r = minimo
    anterior_f = f(anterior_r)

    for k in range(1, passos + 1):
        r = minimo + (maximo - minimo) * k / passos
        fr = f(r)
        if anterior_f == 0:
            return anterior_r
        if anterior_f * fr < 0:
            a, b = anterior_r, r
            fa, fb = anterior_f, fr
            break
        anterior_r, anterior_f = r, fr
    else:
        raise ValueError("Não foi encontrada uma TIR única no intervalo pesquisado.")

    for _ in range(max_iter):
        m = (a + b) / 2
        fm = f(m)
        if abs(fm) < tol or abs(b - a) < tol:
            return m
        if fa * fm <= 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    return (a + b) / 2

def tabela_fluxos(fluxos, taxa):
    linhas = []
    acumulado = 0
    for t, fc in enumerate(fluxos):
        fator = 1 / (1 + taxa)**t
        descontado = fc * fator
        acumulado += descontado
        linhas.append([t, fc, fator, descontado, acumulado])
    return pd.DataFrame(
        linhas,
        columns=["Ano", "Fluxo", "Fator VP", "Fluxo descontado", "Acumulado descontado"]
    )

pd.options.display.float_format = lambda x: f"{x:,.2f}"

## Exercício 1 — VPL da reforma

- Investimento: R$ 50.000
- Fluxo anual: R$ 15.000 por 5 anos
- TMA: 12% a.a.

In [ ]:
investimento = 50_000
fluxo = 15_000
tma = 0.12
n = 5

vpl_ex1 = -investimento + fluxo * fator_vp_serie(tma, n)
print(f"VPL = R$ {vpl_ex1:,.2f}")
print("Decisão:", "Viável" if vpl_ex1 > 0 else "Não viável")

**Resultado:** VPL ≈ **R$ 4.071,64**. Como é positivo, a reforma é viável segundo o critério do VPL.

## Exercício 2 — Escolha entre softwares pelo VPL

In [ ]:
tma = 0.08
fluxos_A = [-80_000] + [25_000] * 4
fluxos_B = [-120_000] + [38_000] * 4

vpl_A = vpl(tma, fluxos_A)
vpl_B = vpl(tma, fluxos_B)

pd.DataFrame({
    "Projeto": ["A", "B"],
    "VPL": [vpl_A, vpl_B]
})

In [ ]:
print(f"VPL A = R$ {vpl_A:,.2f}")
print(f"VPL B = R$ {vpl_B:,.2f}")
print("Escolha:", "B" if vpl_B > vpl_A else "A")

**Resultado:** ambos têm VPL positivo, mas o **Projeto B** possui o maior VPL.

## Exercício 3 — CAUE dos asfaltos

Como só existem custos, comparamos o **custo anual uniforme equivalente**.

In [ ]:
tma = 0.08

caue_comum = 200_000 * fator_recuperacao_capital(tma, 5)
caue_premium = 320_000 * fator_recuperacao_capital(tma, 10)

tabela_caue = pd.DataFrame({
    "Opção": ["Asfalto Comum", "Asfalto Premium"],
    "CAUE": [caue_comum, caue_premium]
})
tabela_caue

In [ ]:
print(f"CAUE Comum   = R$ {caue_comum:,.2f}/ano")
print(f"CAUE Premium = R$ {caue_premium:,.2f}/ano")
print("Mais barato: Asfalto Premium")

## Exercício 4 — TIR

Fluxos:
- Ano 0: -50.000
- Ano 1: +20.000
- Ano 2: +48.000
- TMA: 10%

In [ ]:
fluxos_ex4 = [-50_000, 20_000, 48_000]
tir_ex4 = tir_bissecao(fluxos_ex4)

print(f"TIR = {tir_ex4:.2%}")
print("TMA = 10,00%")
print("Decisão:", "Viável" if tir_ex4 > 0.10 else "Não viável")

**Resultado:** TIR = **20% a.a.**, superior à TMA de 10%.

## Exercício 5 — VPL e TIR incremental

In [ ]:
tma = 0.10
A = [-10_000, 13_000]
B = [-20_000, 24_000]

vpl_A = vpl(tma, A)
vpl_B = vpl(tma, B)

incremental = [B[0] - A[0], B[1] - A[1]]
tir_incremental = tir_bissecao(incremental)

print(f"VPL A = R$ {vpl_A:,.2f}")
print(f"VPL B = R$ {vpl_B:,.2f}")
print("Fluxo incremental B-A:", incremental)
print(f"ΔTIR = {tir_incremental:.2%}")

**Resultado:** os dois VPLs são aproximadamente **R$ 1.818,18**.  
O fluxo incremental é **[-10.000, +11.000]**, cuja ΔTIR é **10%**, exatamente igual à TMA. Logo, o investimento adicional é indiferente segundo esse critério.

## Exercício 6 — Payback simples e descontado

In [ ]:
investimento = 25_000
fc1 = 12_000
fc2 = 18_000
tma = 0.12

# simples
restante_apos_ano1 = investimento - fc1
fracao_ano2_simples = restante_apos_ano1 / fc2
payback_simples = 1 + fracao_ano2_simples

# descontado
vp1 = fc1 / (1 + tma)
vp2 = fc2 / (1 + tma)**2
restante_descontado = investimento - vp1
fracao_ano2_desc = restante_descontado / vp2
payback_desc = 1 + fracao_ano2_desc

print(f"Payback simples = {payback_simples:.4f} anos")
print(f"= 1 ano e {fracao_ano2_simples*12:.2f} meses adicionais")

print(f"Payback descontado = {payback_desc:.4f} anos")
print(f"= 1 ano e {fracao_ano2_desc*12:.2f} meses adicionais")

**Resultado:**
- Payback simples ≈ **1 ano e 8,67 meses**
- Payback descontado ≈ **1 ano e 11,95 meses**, praticamente 2 anos.

## Exercício 7 — Franquia com valor residual

No ano 4, o fluxo é **R$ 30.000 + R$ 10.000 de valor residual = R$ 40.000**.

In [ ]:
tma = 0.15
fluxos_franquia = [-60_000, 30_000, 30_000, 30_000, 40_000]

tabela_franquia = tabela_fluxos(fluxos_franquia, tma)
vpl_franquia = vpl(tma, fluxos_franquia)
tir_franquia = tir_bissecao(fluxos_franquia)

tabela_franquia

In [ ]:
print(f"VPL = R$ {vpl_franquia:,.2f}")
print(f"TIR = {tir_franquia:.2%}")
print("Decisão:", "Viável" if vpl_franquia > 0 else "Não viável")

**Resultado:** VPL ≈ **R$ 31.366,88**, portanto o projeto é viável pela TMA de 15%.

## Exercício 8 — TMA sobe de 15% para 50%

In [ ]:
tma_crise = 0.50

vpl_50 = vpl(tma_crise, fluxos_franquia)
tabela_50 = tabela_fluxos(fluxos_franquia, tma_crise)

print(f"VPL com TMA de 50% = R$ {vpl_50:,.2f}")
print(f"TIR do projeto continua = {tir_franquia:.2%}")
tabela_50

**Interpretação:**
- o **VPL diminui** e fica negativo;
- o **Payback descontado aumenta**; neste horizonte de 4 anos ele deixa de ocorrer;
- a **TIR não muda**, porque depende dos fluxos do projeto, não da TMA usada para avaliá-lo.

## Exercício 9 — Substituição do caminhão

A tabela da aula fornece investimento bruto anual equivalente, valor de revenda equivalente e custos operacionais.

Calculamos:
\[
Investimento\ Líquido = Investimento\ Bruto - Valor\ de\ Revenda
\]

\[
Custo\ Total\ Anual = Investimento\ Líquido + Custos\ Operacionais
\]

In [ ]:
dados = pd.DataFrame({
    "Ano": [1, 2, 3, 4, 5],
    "Investimento bruto": [737000.00, 386047.62, 269416.92, 211365.44, 176744.31],
    "Valor de revenda": [569500.00, 235238.10, 128700.91, 70758.95, 40212.28],
    "Custos operacionais": [10000.00, 12380.95, 16344.41, 23262.23, 35323.91],
})

dados["Investimento líquido"] = dados["Investimento bruto"] - dados["Valor de revenda"]
dados["Custo total anual"] = dados["Investimento líquido"] + dados["Custos operacionais"]

dados

In [ ]:
idx = dados["Custo total anual"].idxmin()
melhor_ano = int(dados.loc[idx, "Ano"])
menor_custo = dados.loc[idx, "Custo total anual"]

print(f"Menor custo total anual = R$ {menor_custo:,.2f}")
print(f"Vida econômica = {melhor_ano} anos")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(dados["Ano"], dados["Custo total anual"], marker="o")
plt.title("Custo Total Anual do Caminhão")
plt.xlabel("Tempo de permanência (anos)")
plt.ylabel("Custo total anual (R$)")
plt.grid(True, alpha=0.25)
plt.show()

**Resultado:** o custo total anual é mínimo no **Ano 3**, aproximadamente **R$ 157.060,42/ano**.  
Portanto, segundo o critério da aula, o caminhão deve ser mantido por **3 anos**.

## Resumo final

| Exercício | Tema | Resultado principal |
|---|---|---|
| 1 | VPL | R$ 4.071,64 — viável |
| 2 | VPL A x B | Projeto B |
| 3 | CAUE | Asfalto Premium |
| 4 | TIR | 20% > 10% |
| 5 | TIR incremental | ΔTIR = 10% = TMA |
| 6 | Payback | Simples ≈ 1,72 anos; descontado ≈ 2 anos |
| 7 | Valor residual | VPL ≈ R$ 31.366,88 |
| 8 | Sensibilidade | TMA ↑ → VPL ↓ e payback descontado ↑; TIR não muda |
| 9 | Substituição | Vida econômica = 3 anos |

### Observação importante do Excel
A aula alerta que `VPL()` no Excel desconta os fluxos futuros, mas **não inclui o fluxo do momento zero**. Portanto, o investimento inicial deve ser somado separadamente:
`=VPL(taxa; fluxos_1_a_n) + FC0`.